In [ ]:
import numpy as np
import pandas as pd

from dowhy import CausalModel

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# ----------------------------
# 3) Create a realistic dataset
# ----------------------------
# In real life, you'd load a dataset like:
#   df = pd.read_parquet("events_aggregated_weekly.parquet")
#
# Here we simulate common product analytics confounding:
#   - more engaged users are more likely to enable personalization
#   - mobile users behave differently
#   - traffic source influences both feature exposure and engagement

N = 25_000

# ----------------------------
# 3a) User attributes (upstream variables)
# ----------------------------
# tenure_days: "how many days since signup" at time of observation.
# age: age of the user in years (positive integer).
#
# We force both to be positive integers (as in your earlier revision).

tenure_days = rng.gamma(shape=2.0, scale=120.0, size=N)  # positive continuous
tenure_days = np.clip(np.rint(tenure_days).astype(int), 1, None)  # positive int

# age: simulate a plausible adult distribution (16–85), with more mass in mid-adulthood.
age = rng.normal(loc=40, scale=15, size=N)  # continuous, could be outside bounds
age = np.clip(np.rint(age).astype(int), 16, 85)  # positive int, bounded to realistic range

# A helper: normalize a variable for stable coefficient scaling
# (Even though some columns are integers, it is completely fine to standardize them for modelling.)
def z(x):
    return (x - np.mean(x)) / (np.std(x) + 1e-9)

# ----------------------------
# 3b) Device (age -> is_mobile)
# ----------------------------
# In the DAG: age -> is_mobile
# Interpretation: older users are (slightly) less likely to be primarily mobile.
#
# We generate is_mobile via a logistic model so age meaningfully influences device choice.
logit_mobile = 0.7 - 0.35 * z(age)  # higher age -> slightly lower probability of being mobile
p_mobile = 1 / (1 + np.exp(-logit_mobile))
is_mobile = rng.binomial(1, p_mobile, size=N)

# ----------------------------
# 3c) Acquisition channel (age -> traffic_social)
# ----------------------------
# In the DAG: age -> traffic_social
# Interpretation: older users tend to arrive less via social media.
#
# We generate mutually exclusive traffic_source categories, then one-hot encode them.
# To make age affect social traffic, we let P(social) decrease with age.

# Base probabilities (before age adjustment)
# We'll adjust only social probability with age, then renormalize.
base_p_search = 0.45
base_p_social = 0.25
base_p_direct = 0.30

# Create an age-adjusted social probability:
# - younger users: higher social probability
# - older users: lower social probability
# We'll keep it bounded to avoid extreme values.
p_social = base_p_social - 0.10 * z(age)                 # decrease with age
p_social = np.clip(p_social, 0.05, 0.45)                 # keep probabilities sensible

# Split the remaining probability mass between search and direct in fixed proportions.
# (This is just one reasonable approach for simulation.)
remaining = 1.0 - p_social
ratio_search = base_p_search / (base_p_search + base_p_direct)
p_search = remaining * ratio_search
p_direct = remaining * (1 - ratio_search)

# Sanity: ensure probabilities sum to 1 for every user.
# (Floating-point error can happen; this protects multinomial sampling.)
prob_sum = p_search + p_social + p_direct
p_search, p_social, p_direct = p_search / prob_sum, p_social / prob_sum, p_direct / prob_sum

# Sample a categorical traffic source using user-specific probabilities.
u = rng.random(size=N)
traffic_source = np.where(
    u < p_search, "search",
    np.where(u < (p_search + p_social), "social", "direct")
)

# One-hot encode traffic source (keeps things numeric & simple for DoWhy)
traffic_search = (traffic_source == "search").astype(int)
traffic_social = (traffic_source == "social").astype(int)
traffic_direct = (traffic_source == "direct").astype(int)

# ----------------------------
# 3d) Baseline engagement (tenure, traffic, device -> prior_week_minutes)
# ----------------------------
# In the DAG:
#   tenure_days -> prior_week_minutes
#   traffic_*   -> prior_week_minutes
#   is_mobile   -> prior_week_minutes
#
# We generate a *continuous positive* baseline engagement signal, then convert to
# positive integers (minutes).
#
# We model baseline engagement on the log scale to keep positivity natural.
# Intuition:
# - longer tenure users tend to have higher baseline engagement
# - direct users are more habitual (higher baseline), social users more episodic (lower baseline)
# - mobile users might have slightly different baseline usage
noise_prior = rng.normal(0, 0.6, size=N)  # log-scale noise; controls dispersion

log_prior = (
    2.7
    + 0.35 * z(np.log1p(tenure_days))
    + 0.25 * traffic_direct
    - 0.15 * traffic_social
    + 0.05 * traffic_search
    + 0.08 * is_mobile
    + noise_prior
)

prior_week_minutes = np.exp(log_prior)  # positive continuous

# Force prior_week_minutes to be a positive integer.
# - rint() rounds to nearest integer
# - clip() ensures strictly positive (>= 1)
prior_week_minutes = np.clip(np.rint(prior_week_minutes).astype(int), 1, None)

# ----------------------------
# 3e) Treatment assignment mechanism (observational, confounded)
# ----------------------------
# Treatment T = 1 if personalization is enabled/exposed.
#
# In the DAG:
#   prior_week_minutes -> personalized_homepage
#   is_mobile -> personalized_homepage
#   traffic_direct -> personalized_homepage
#
# Note what we do NOT do (to stay aligned with the DAG):
# - We do NOT put tenure_days directly into treatment.
#   Tenure influences treatment only through prior_week_minutes (mediated path).
# - We do NOT put age directly into treatment.
#   Age influences treatment only through is_mobile (and indirectly through channel/engagement).
#
# We use log1p() to stabilize skewed positive variables.
logit_p = (
    -0.2
    + 0.95 * z(np.log1p(prior_week_minutes))
    + 0.30 * is_mobile
    + 0.25 * traffic_direct
)

p_treat = 1 / (1 + np.exp(-logit_p))
personalized_homepage = rng.binomial(1, p_treat, size=N)  # treatment T

# ----------------------------
# 3f) Outcome generation mechanism (true causal effect + confounding)
# ----------------------------
# Outcome Y = weekly engagement minutes.
#
# In the DAG:
#   personalized_homepage -> weekly_minutes  (causal effect)
#   prior_week_minutes -> weekly_minutes
#   is_mobile -> weekly_minutes
#   traffic_* -> weekly_minutes
#   age -> weekly_minutes  (optional; included in our DAG)
#
# We'll set a TRUE average treatment effect (ATE) to validate learning.
TRUE_ATE = 6.0  # personalization adds +6 minutes/week on average (in our simulation)

# Add noise. This can make weekly_minutes non-integer and even negative pre-clipping.
noise = rng.normal(0, 10, size=N)

weekly_minutes = (
    25
    + TRUE_ATE * personalized_homepage                      # <-- causal effect of interest
    + 8.5 * z(np.log1p(prior_week_minutes))                 # baseline habit persists
    + 2.0 * is_mobile                                       # device context effect
    + 2.5 * traffic_direct - 1.0 * traffic_social + 0.3 * traffic_search  # channel intent effect
    - 1.0 * z(age)                                          # optional: older users slightly less time
    + noise
)

# Force weekly_minutes to be a positive integer.
# - rint() rounds to nearest integer
# - clip() ensures strictly positive (>= 1)
weekly_minutes = np.clip(np.rint(weekly_minutes).astype(int), 1, None)

# ----------------------------
# 3g) Assemble dataset
# ----------------------------
df = pd.DataFrame(
    {
        # Treatment and outcome
        "personalized_homepage": personalized_homepage,
        "weekly_minutes": weekly_minutes,          # positive int
        # Upstream user attributes (not necessarily "direct confounders" in the DAG)
        "tenure_days": tenure_days,                # positive int
        "age": age,                                # positive int
        # Baseline engagement proxy (pre-treatment)
        "prior_week_minutes": prior_week_minutes,  # positive int
        # Context covariates
        "is_mobile": is_mobile,
        "traffic_search": traffic_search,
        "traffic_social": traffic_social,
        "traffic_direct": traffic_direct,
    }
)

# ----------------------------
# 3h) Quick sanity checks
# ----------------------------
# Check data types and positivity constraints explicitly (best practice)
assert (df["tenure_days"] > 0).all(), "tenure_days must be strictly positive"
assert (df["age"] > 0).all(), "age must be strictly positive"
assert (df["prior_week_minutes"] > 0).all(), "prior_week_minutes must be strictly positive"
assert (df["weekly_minutes"] > 0).all(), "weekly_minutes must be strictly positive"

assert pd.api.types.is_integer_dtype(df["tenure_days"]), "tenure_days must be integer dtype"
assert pd.api.types.is_integer_dtype(df["age"]), "age must be integer dtype"
assert pd.api.types.is_integer_dtype(df["prior_week_minutes"]), "prior_week_minutes must be integer dtype"
assert pd.api.types.is_integer_dtype(df["weekly_minutes"]), "weekly_minutes must be integer dtype"

print("Dataset preview:")
display(df.head())

print("\nDescriptive statistics:")
display(df.describe())

print("\nTreatment rate:", df["personalized_homepage"].mean().round(3))
print("Outcome mean:", df["weekly_minutes"].mean().round(2))

# Naive difference in means (confounded!)
naive_diff = (
    df.loc[df.personalized_homepage == 1, "weekly_minutes"].mean()
    - df.loc[df.personalized_homepage == 0, "weekly_minutes"].mean()
)
print("\nNaive (confounded) diff-in-means:", round(naive_diff, 2), "minutes/week")
print("True ATE (in simulation):", TRUE_ATE, "minutes/week")


In [ ]:
#Python 3.14.2
#I. Model the causal question. 

from dowhy import CausalModel

# Define a causal graph (DAG)
causal_graph = """
digraph {
    tenure_days -> prior_week_minutes;
    age -> traffic_social;
    age -> is_mobile;
    traffic_direct -> prior_week_minutes;
    traffic_social -> prior_week_minutes;
    traffic_search -> prior_week_minutes;
    is_mobile -> prior_week_minutes;
    prior_week_minutes -> personalized_homepage;
    is_mobile -> personalized_homepage;
    traffic_direct -> personalized_homepage;
    personalized_homepage -> weekly_minutes;
    prior_week_minutes -> weekly_minutes;
    is_mobile -> weekly_minutes;
    traffic_direct -> weekly_minutes;
    traffic_social -> weekly_minutes;
    traffic_search -> weekly_minutes;
    age -> weekly_minutes;
}
"""

In [ ]:
# Create DoWhy CausalModel
model = CausalModel(
    data=df,
    treatment="personalized_homepage",
    outcome="weekly_minutes",
    graph=causal_graph
)

# Visualize the DAG
model.view_model()

In [ ]:
import networkx as nx

#II. Identify the causal estimand. 
#Next, we ask DoWhy to identify the causal effect. 
#This step does not estimate the effect yet. Instead, it answers a question:
#Under what conditions can the causal effect of the treatment on the outcome be identified from the data?
#An estimand is a mathematical expression of the causal quantity we want, such as the Average Treatment Effect (ATE):


# Compatibility patch: NetworkX 3.x renamed d_separated -> is_d_separator
# DoWhy still calls nx.algorithms.d_separated, so we alias the new function name.
if not hasattr(nx.algorithms, "d_separated"):
    from networkx.algorithms.d_separation import is_d_separator
    nx.algorithms.d_separated = is_d_separator

identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
print("Identified estimand:")
print(identified_estimand)


#Block all backdoor paths from treatment to outcome by conditioning on observed variables.
#Once we condition on these variables, the remaining difference in Y can be interpreted as causal. 
#These variables form a valid adjustment set that blocks all backdoor paths in the DAG.
#Now that we know what to control for, we can actually estimate the causal effect.
#Controlling means comparing persons with very similar values of the confounders. Common approaches include:
#Linear regression; Propensity score matching; Propensity score weighting


In [ ]:
#III. Estimate the causal effect - linear regression.

# Linear regression adjustment (backdoor)
estimate_lr = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
    test_significance=True
)
print("[Estimate] Backdoor linear regression:")
print(estimate_lr)
print("ATE estimate:", round(estimate_lr.value, 3))

#The output shows an estimated average treatment effect (ATE) of about 7.56 minutes.
#Causal validity comes from the DAG assumptions.

In [ ]:
#III. Estimate the causal effect - matching.

#Estimate each user's probability of receiving treatment.
#Match treated and control users with similar propensity scores.
#Compare their outcomes.

# Propensity score matching (backdoor)
estimate_psm = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_matching",
    target_units="ate",
)

print("[Estimate] Propensity score matching:")
print(estimate_psm)
print("ATE estimate:", round(estimate_psm.value, 3))

#5.24 (95% CI 3.39 and 6.06)


In [ ]:
#III. Estimate the causal effect -IPTW
#This method keeps all units but reweights them to create a synthetically balanced population. 

# Propensity score weighting (backdoor)
estimate_psw = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_weighting",
    target_units="ate",
)

print("[Estimate] Propensity score weighting:")
print(estimate_psw)
print("ATE estimate:", round(estimate_psw.value, 3))

#5.995 


In [ ]:
#IV. Refute the obtained estimate. 
#All refutation tests operate within the assumptions encoded in the Directed Acyclic Graph (DAG). 
#If the DAG itself is wrong, for example if an important confounder is missing, then the causal estimate may still be incorrect even if every refutation test passes.

#IV. Refute the obtained estimate - Random Common Cause

# Add a random common cause (should not change estimate much)
refute_random_cc = model.refute_estimate(
    identified_estimand,
    estimate_psw,
    method_name="random_common_cause"
)
print("[Refute] Random common cause:")
print(refute_random_cc)

In [ ]:
#IV. Refute the obtained estimate - Placebo Treatment Refuter
#Interestingly, this refuter does NOT work well with propensity score weighting.

# Placebo treatment (permute treatment) -> effect should go ~0
refute_placebo = model.refute_estimate(
    identified_estimand,
    estimate_lr,
    method_name="placebo_treatment_refuter",
    placebo_type="permute"
)
print("[Refute] Placebo treatment on Linear Regression (permute):")
print(refute_placebo)

In [ ]:
#IV. Refute the obtained estimate - Data Subset Refuter
# Data subset refuter: estimate on random subsets -> should be stable
refute_subset = model.refute_estimate(
    identified_estimand,
    estimate_lr,
    method_name="data_subset_refuter",
    subset_fraction=0.8,
    num_simulations=20,
    random_state=RANDOM_SEED
)
print("[Refute] Data subset refuter:")
print(refute_subset)


In [ ]:
#IV. Refute the obtained estimate - bootstrap Refuter

# Bootstrap refuter: resample with replacement -> variability/robustness
refute_bootstrap = model.refute_estimate(
    identified_estimand,
    estimate_lr,
    method_name="bootstrap_refuter",
    num_simulations=10,
    random_state=RANDOM_SEED
)
print("\n[Refute] Bootstrap refuter:")
print(refute_bootstrap)


In [ ]:
#IV. Refute the obtained estimate - Unobserved Common Cause Refuter

refute_uc = model.refute_estimate(
    identified_estimand,
    estimate_lr,
    method_name="add_unobserved_common_cause",
    confounders_effect_on_treatment="binary_flip",
    confounders_effect_on_outcome="linear",
    effect_strength_on_treatment=0.05,
    effect_strength_on_outcome=0.05,
    num_simulations=50, # Optional
    random_state=42
)

print(refute_uc)

In [ ]:
#Diagnostics


# Fit a propensity model ourselves (for diagnostics)
X = df[
    [
        "tenure_days", "prior_week_minutes", "is_mobile",
        "age", "traffic_search", "traffic_social", "traffic_direct"
    ]
]
t = df["personalized_homepage"]

X_train, X_test, t_train, t_test = train_test_split(X, t, test_size=0.3, random_state=RANDOM_SEED, stratify=t)

ps_model = LogisticRegression(max_iter=2000)
ps_model.fit(X_train, t_train)
propensity_scores = ps_model.predict_proba(X)[:, 1]
df["propensity"] = propensity_scores

print("\nPropensity summary (treated vs control):")
print(df.groupby("personalized_homepage")["propensity"].describe()[["mean", "std", "min", "max"]])

# Overlap check
plt.figure()
df[df.personalized_homepage == 1]["propensity"].hist(bins=30, alpha=0.6, label="Treated")
df[df.personalized_homepage == 0]["propensity"].hist(bins=30, alpha=0.6, label="Control")
plt.title("Propensity score overlap diagnostic")
plt.xlabel("Estimated P(Treatment=1 | confounders)")
plt.ylabel("Count")
plt.legend()
plt.show()

In [ ]:
#SMd

def smd(x_treated, x_control):
    """Compute standardized mean difference."""
    mean_t = np.mean(x_treated)
    mean_c = np.mean(x_control)
    var_t = np.var(x_treated, ddof=1)
    var_c = np.var(x_control, ddof=1)
    return (mean_t - mean_c) / np.sqrt((var_t + var_c) / 2)


# Pre-treatment SMD (before any adjustment)
covariates = [
    "tenure_days",
    "prior_week_minutes",
    "is_mobile",
    "age",
    "traffic_search",
    "traffic_social",
    "traffic_direct",
]

smd_pre = {}

for col in covariates:
    smd_pre[col] = smd(
        df.loc[df.personalized_homepage == 1, col],
        df.loc[df.personalized_homepage == 0, col],
    )

smd_pre_df = (
    pd.DataFrame.from_dict(smd_pre, orient="index", columns=["SMD_pre"])
    .sort_values("SMD_pre", key=np.abs, ascending=False)
)

print("\nPre-treatment SMD:")
display(smd_pre_df)


In [ ]:
# Step 1: Separate treated and control units
treated = df[df.personalized_homepage == 1].copy()
control = df[df.personalized_homepage == 0].copy()

# Step 2: Nearest-neighbor matching on propensity score
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=1)
nn.fit(control[["propensity"]])

distances, indices = nn.kneighbors(treated[["propensity"]])

# Step 3: Apply caliper (distance threshold)
caliper = 0.05
mask = distances.flatten() <= caliper

# Keep only well-matched treated units
treated_matched = treated.loc[mask].copy()

# Select matched control units
matched_control = control.iloc[indices.flatten()].copy()
matched_control = matched_control.loc[mask].copy()

# Optional: align indices (helps with diagnostics later)
matched_control.index = treated_matched.index

# Step 4: Construct matched dataset
df_psm = pd.concat([treated_matched, matched_control], axis=0)


smd_psm = {}

for col in covariates:
    smd_psm[col] = smd(
        df_psm.loc[df_psm.personalized_homepage == 1, col],
        df_psm.loc[df_psm.personalized_homepage == 0, col],
    )

smd_psm_df = (
    pd.DataFrame.from_dict(smd_psm, orient="index", columns=["SMD_post_PSM"])
    .sort_values("SMD_post_PSM", key=np.abs, ascending=False)
)

print("\nPost-PSM SMD:")
display(smd_psm_df)

In [ ]:
#export df as a Stata dataset
df.to_stata("C:/Causal_Inference/Treatment_effects/outputs/df.dta", write_index=False)
#use "C:\Python\Causal_Inference\DoWhy\outputs\df.dta"
#rename personalized_homepage treat
#rename weekly_minutes outcome
#teffects ra (outcome tenure_days age traffic_social) (treat), atet nolog
